In [ ]:
# New CODE
# Install required libraries
!pip install opencv-python-headless numpy matplotlib heapq

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import random
import heapq

# Step 1: Upload PNG file
uploaded = files.upload()
img_filename = list(uploaded.keys())[0]

# Step 2: Read image and preprocess
raster_img = cv2.imread(img_filename, cv2.IMREAD_GRAYSCALE)
blurred = cv2.GaussianBlur(raster_img, (5, 5), 0)
edges = cv2.Canny(blurred, 50, 150)

# Step 3: Detect contours as obstacles
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
raster_colored = cv2.cvtColor(raster_img, cv2.COLOR_GRAY2BGR)
cv2.drawContours(raster_colored, contours, -1, (0, 255, 0), 1)

# Step 4: Obstacle mask creation
obstacle_mask = np.zeros_like(raster_img, dtype=np.uint8)
cv2.drawContours(obstacle_mask, contours, -1, (255), thickness=cv2.FILLED)

# Step 5: Define start and goal
height, width = raster_img.shape
start = (height - 10, width // 2)
end = (10, width // 2)

# Node definition for RRT + HGS
class Node:
    def __init__(self, pos, parent=None):
        self.pos = pos
        self.parent = parent

# Distance heuristic
def dist(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# Collision checking
def collision_free(pos):
    x, y = pos
    return 0 <= x < height and 0 <= y < width and obstacle_mask[int(x), int(y)] == 0

# Hunger Games Optimization influenced sampling
def hgs_sample(goal, best_pos, W_adaptive):
    if random.random() < W_adaptive:
        direction = np.array(goal) - np.array(best_pos)
    else:
        direction = np.random.uniform(-1, 1, size=2)
    direction = direction / np.linalg.norm(direction)
    sample_point = np.array(best_pos) + direction * random.randint(10, 30)
    sample_point = np.clip(sample_point, [0,0], [height-1,width-1])
    return tuple(sample_point.astype(int))

# RRT + Hunger Games Search (HGS) algorithm implementation
def hgs_rrt(start, end, iterations=5000):
    tree = [Node(start)]
    best_node = tree[0]

    for i in range(iterations):
        W_adaptive = i / iterations
        rand_point = hgs_sample(end, best_node.pos, W_adaptive)
        nearest = min(tree, key=lambda node: dist(node.pos, rand_point))

        direction = np.array(rand_point) - np.array(nearest.pos)
        direction = direction / np.linalg.norm(direction)
        new_pos = tuple((np.array(nearest.pos) + direction * 5).astype(int))

        if collision_free(new_pos):
            new_node = Node(new_pos, nearest)
            tree.append(new_node)

            if dist(new_pos, end) < dist(best_node.pos, end):
                best_node = new_node

            if dist(new_pos, end) < 10:
                return new_node

    return best_node

# Execute RRT+HGS
final_node = hgs_rrt(start, end)

# Path extraction and visualization
if final_node:
    path_coords = []
    while final_node:
        path_coords.append(final_node.pos)
        final_node = final_node.parent

    for coord in path_coords:
        y, x = coord
        cv2.circle(raster_colored, (x, y), 2, (0, 0, 255), -1)

# Display results
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(raster_colored, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Safe Path using RRT + Hunger Games Optimization")
plt.show()

# Save and download
output_filename = "safe_hgs_rrt_path.png"
cv2.imwrite(output_filename, raster_colored)
files.download(output_filename)

KeyboardInterrupt: 